Heuristics
==========

**Author:** Rafael



## Uniform graph partition



En este problema, la instancia es una gráfica con $2n$ vértices y con pesos en las aristas. El objetivo es encontrar una partición $X,Y$ del conjunto de los vértices de $G$, tal que $|X|=|Y|=n$ de tal manera que minimice la suma de los pesos de las aristas que tienen un extremo en $X$ y el otro extremo en $Y$.



### Instancia



In [1]:
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import networkx as nx
import math
import random

@dataclass
class WeightedGraphInstance:
    """A complete weighted graph for graph-partition problems.
    Nodes are labelled 0 to num_vertices - 1. Each edge carries a 'weight'.
    num_vertices must be even (two equal-sized partitions are expected).
    """
    num_vertices: int
    max_weight: int = 10
    graph: nx.Graph = field(init=False, repr=False)

    def __post_init__(self):
        self.graph = nx.complete_graph(self.num_vertices)
        for u, v in self.graph.edges():
            self.graph[u][v]['weight'] = random.randint(0, self.max_weight)

    def draw(self):
        """Draws the graph with edge weights displayed."""
        pos = nx.circular_layout(self.graph)
        nx.draw(self.graph, pos, with_labels=True, node_color='lightblue',
                node_size=500, font_weight='bold')
        edge_labels = nx.get_edge_attributes(self.graph, 'weight')
        nx.draw_networkx_edge_labels(self.graph, pos, edge_labels=edge_labels)
        plt.show()

    def get_cost(self, u, v):
        if u == v:
            return 0
        return self.graph[u][v]['weight']

In [1]:
G = WeightedGraphInstance(100)
#G.draw()

In [1]:
G.get_cost(0, 2)

### Funciones



In [1]:
def random_equal_partition(n):
    """
    Returns a random partition of the nodes {0..2n-1} into two subsets of
    size n each. (Algorithm 5.7)
    """
    all_nodes = list(range(2 * n))
    random.shuffle(all_nodes)
    return [sorted(all_nodes[:n]), sorted(all_nodes[n:])]

parti = random_equal_partition(50)
parti

In [1]:
def partition_cost(instance, partition):
    """
    Returns the total weight of edges crossing the partition in the weighted
    nx.Graph. partition is a pair of lists of 0-indexed node labels.
    """
    cost = 0
    for u in partition[0]:
        for v in partition[1]:
            cost += instance.get_cost(u, v)
    return cost

partition_cost(G, parti)

In [1]:
def partition_gain(instance, partition, vertex_from_first, vertex_from_second):
    """
    Returns the change in partition_cost when swapping vertex_from_first
    (currently in partition[0]) with vertex_from_second (currently in
    partition[1]) in the weighted nx.Graph.
    """
    graph = instance.graph
    u = vertex_from_first
    v = vertex_from_second

    gain = 0
    for x in partition[0]:
        gain += instance.get_cost(x, v)
    for x in partition[0]:
        gain -= instance.get_cost(x, u)
    for y in partition[1]:
        gain += instance.get_cost(u, y)
    for y in partition[1]:
        gain -= instance.get_cost(v, y)
    gain -= 2 * instance.get_cost(u, v)
    return gain

partition_gain(G, parti, 4, 0)

### Encontrar la mejor partición



In [1]:
def partition_ascent_step(instance, partition):
    """
    Given a partition of the vertices of a weighted graph, attempts a single
    improving swap. Returns [new_partition, is_local_optimum, gain_achieved].
    """
    current = [list(partition[0]), list(partition[1])]
    best_gain = 0
    best_u = best_v = None
    for u in current[0]:
        for v in current[1]:
            gain = partition_gain(instance, current, u, v)
            if gain > best_gain:
                best_u, best_v, best_gain = u, v, gain
    if best_gain > 0:
        current[0] = sorted((set(current[0]) - {best_u}) | {best_v})
        current[1] = sorted((set(current[1]) - {best_v}) | {best_u})
        return [current, False, best_gain]
    else:
        return [current, True, best_gain]

partition_ascent_step(G, parti)

In [1]:
def uniform_graph_partition(instance, max_iterations):
    """
    Solves the uniform graph partition problem for a weighted nx.Graph using
    hill-climbing for at most max_iterations iterations.
    """
    graph = instance.graph
    n = graph.number_of_nodes() // 2
    partition = random_equal_partition(n)
    for i in range(max_iterations):
        result = partition_ascent_step(instance, partition)
        if result[1]:  # local optimum reached
            print(f"Local optimum! (after {i} iterations)")
            return result[0]
        partition = result[0]
    return partition

In [1]:
parti=uniform_graph_partition(G, 100)

In [1]:
partition_cost(G, parti)

## With Simulated Annealing



Recocido simulado



In [1]:
import math
import random

def uniform_graph_partition_sa(instance, max_iterations, T0, alpha):
    """
    Solves the uniform graph partition problem using Simulated Annealing.
    """
    c = 0  # iteration counter
    T = T0 # current temperature
    
    # Select a feasible solution X (partition)
    graph = instance.graph
    n = graph.number_of_nodes() // 2
    partition = random_equal_partition(n)
    
    # Track the best solution found (X_best)
    best_partition = (partition[0][:], partition[1][:])
    
    # Pre-calculate current cost (P(X)) to track improvements
    current_cost = partition_cost(instance, partition)
    best_cost = current_cost

    while c <= max_iterations: # while c <= cmax
        # 1. Generate a neighbor Y by swapping random elements (h_N(X))
        u = random.choice(partition[0])
        v = random.choice(partition[1])
        
        # Calculate the gain: P(Y) - P(X)
        # In this context, gain = current_cost - neighbor_cost 
        gain = partition_gain(instance, partition, u, v)
        
        # In the algorithm: if P(Y) > P(X)
        # For cost minimization, this means the gain is positive (cost decreases)
        if gain > 0:
            # Update partition (X <- Y)
            partition[0].remove(u)
            partition[0].append(v)
            partition[1].remove(v)
            partition[1].append(u)
            current_cost -= gain
            
            # if P(X) > P(X_best)
            if current_cost < best_cost:
                best_partition = (partition[0][:], partition[1][:])
                best_cost = current_cost
        else:
            # Probabilistic acceptance for worse moves (else block)
            r = random.random() # Random(0, 1)
            
            # Acceptance: r < e^((P(Y) - P(X)) / T)
            # Since gain is P(Y) - P(X), we use: r < e^(gain / T)
            if r < math.exp(gain / T):
                partition[0].remove(u)
                partition[0].append(v)
                partition[1].remove(v)
                partition[1].append(u)
                current_cost -= gain
                print("Accepted")
            print(f"Probability of acceptance: {math.exp(gain / T)}")    

        # 2. Update state
        c += 1      
        T = alpha * T 

    return best_partition

In [1]:
parti=uniform_graph_partition_sa(G, 100, 8000, 0.999)

In [1]:
partition_cost(G, parti)

In [1]:
import math
import random

def uniform_graph_partition_hybrid_sa(instance, max_iterations, T0, alpha):
    """
    Solves the uniform graph partition problem using an exhaustive ascent
    strategy combined with Simulated Annealing escapes (Algorithm 5.3).
    """
    # Initialize variables based on Algorithm 5.3
    c = 0  # iteration counter
    T = T0 # current temperature
    
    # Select a feasible initial solution X (partition)
    graph = instance.graph
    n = graph.number_of_nodes() // 2
    partition = random_equal_partition(n)
    
    # X_best <- X
    best_partition = (partition[0][:], partition[1][:])
    
    # We track the current cut weight to determine the best overall solution
    best_cost = partition_cost(instance, best_partition)

    while c <= max_iterations: # while c <= cmax
        # Attempt an exhaustive improving step (Greedy Ascent)
        # result = [new_partition, is_local_optimum, gain_achieved]
        result = partition_ascent_step(instance, partition)
        
        # if P(Y) > P(X)
        if not result[1]: # Not a local optimum; an improving move was found
            partition = result[0]
            
            # if P(X) > P(X_best) then X_best <- X
            current_cost = partition_cost(instance, partition)
            if current_cost < best_cost:
                best_partition = (partition[0][:], partition[1][:])
                best_cost = current_cost
        
        else: # is_local_optimum is True 
            # Perform a random swap to find a "worst" neighbor for escape
            u = random.choice(partition[0])
            v = random.choice(partition[1])
            
            # gain = P(Y) - P(X)
            gain = partition_gain(instance, partition, u, v)
            
            # r <- Random(0, 1)
            r = random.random()
            
            # if r < e^((P(Y) - P(X)) / T)
            # (Note: gain here is negative or zero since we are at a local optimum)
            if r < math.exp(gain / T):
                partition[0].remove(u)
                partition[0].append(v)
                partition[1].remove(v)
                partition[1].append(u)
                print("Accepted")
            print(f"Probability of acceptance: {math.exp(gain / T)}")                  

        # Update counter and temperature
        c += 1          
        T = alpha * T   

    return best_partition

In [1]:
parti=uniform_graph_partition_hybrid_sa(G, 100, 100, 0.95)

In [1]:
partition_cost(G, parti)

## With Tabu Search



In [1]:
from collections import deque

my_tabu = deque(maxlen=3)
dir(my_tabu)

In [1]:
my_tabu.append("fresa")
my_tabu.append("limón")
my_tabu.append("sandía")
my_tabu

In [1]:
my_tabu.append("hola")
my_tabu

In [1]:
"fresa" in my_tabu, "limón" in my_tabu

In [1]:
import math
import random
from collections import deque

def uniform_graph_partition_tabu_sa(instance, max_iterations, T0, alpha, tabu_tenure):
    """
    Solves the uniform graph partition problem by combining Tabu Search (Algorithm 5.4)
    with Simulated Annealing (Algorithm 5.3).
    """
    # 1. Initialization (c <- 0 for SA, c <- 1 for Tabu)
    c = 1
    T = T0
    
    # Select a feasible solution X
    graph = instance.graph
    n = graph.number_of_nodes() // 2
    partition = random_equal_partition(n)
    
    # X_best <- X
    best_partition = [list(partition[0]), list(partition[1])]
    best_cost = partition_cost(instance, best_partition)
    
    # Initialize Tabu List with tenure L
    tabu_list = deque(maxlen=tabu_tenure)

    # while c <= c_max
    while c <= max_iterations:
        best_neighbor_gain = -float('inf')
        best_neighbor_move = None
        
        # N <- N(X) \ {TabuList}
        # We find the best move among all non-tabu neighbors
        for u in partition[0]:
            for v in partition[1]:
                current_move = frozenset([u, v])
                
                if current_move in tabu_list:
                    continue
                
                # Calculate gain: P(Y) - P(X)
                gain = partition_gain(instance, partition, u, v)
                
                if gain > best_neighbor_gain:
                    best_neighbor_gain = gain
                    best_neighbor_move = (u, v)

        # if N = empty then exit
        if best_neighbor_move is None:
            break

        u_swap, v_swap = best_neighbor_move
        
        # if P(Y) > P(X)
        if best_neighbor_gain > 0:
            # Always accept improving moves
            partition[0].remove(u_swap)
            partition[0].append(v_swap)
            partition[1].remove(v_swap)
            partition[1].append(u_swap)
            
            # Update best found (X_best <- X)
            current_cost = partition_cost(instance, partition)
            if current_cost < best_cost:
                best_partition = [list(partition[0]), list(partition[1])]
                best_cost = current_cost
        else:
            # Apply Simulated Annealing probabilistic acceptance for worse moves
            r = random.random()
            # if r < e^((P(Y) - P(X)) / T)
            if r < math.exp(best_neighbor_gain / T):
                partition[0].remove(u_swap)
                partition[0].append(v_swap)
                partition[1].remove(v_swap)
                partition[1].append(u_swap)

        # TabuList[c] <- change(Y, X)
        tabu_list.append(frozenset([u_swap, v_swap]))

        # Update counter and temperature
        c += 1
        T = alpha * T

    return best_partition

In [1]:
parti = uniform_graph_partition_tabu_sa(G, 200, 100, 0.99, 10)

In [1]:
partition_cost(G, parti)

## SA for Knapsack



In [1]:
import math
import random
from dataclasses import dataclass
from typing import List

@dataclass
class KnapsackInstance:
    profits: List[float]
    weights: List[float]
    capacity: float

def knapsack_simulated_annealing(instance: KnapsackInstance,
                                 max_iterations: int,
                                 initial_temp: float,
                                 cooling_rate: float):
    """
    Searches for an optimal Knapsack solution using simulated annealing
    """
    profits = instance.profits
    weights = instance.weights
    capacity = instance.capacity
    n = len(profits)
    
    current = [0] * n
    current_weight = 0
    current_profit = 0  
    
    # Initialize the dictionary from the beginning
    best = {
        "selection": list(current),
        "profit": 0
    }
    
    temperature = initial_temp
    
    for _ in range(max_iterations + 1):
        flip_idx = random.randint(0, n - 1)
        
        # Scenario A: Item is currently NOT in the knapsack (we try to add it)
        if current[flip_idx] == 0:
            # Only add if it does not violate the capacity constraint
            if current_weight + weights[flip_idx] <= capacity:
                current[flip_idx] = 1
                current_weight += weights[flip_idx]
                current_profit += profits[flip_idx]
                
                # Check against the dictionary's profit and update directly
                if current_profit > best["profit"]:
                    best["selection"] = list(current)
                    best["profit"] = current_profit
                    
        # Scenario B: Item IS currently in the knapsack (we try to remove it)
        else:
            # Removing an item is always a "worse" move since it lowers profit.
            # We accept this degradation with a probability based on the temperature.
            if random.random() < math.exp(-profits[flip_idx] / temperature):
                current[flip_idx] = 0
                current_weight -= weights[flip_idx]
                current_profit -= profits[flip_idx]
                
        # Cool down the system
        temperature *= cooling_rate
        
    return best

In [1]:
weights = [15, 12, 10, 8, 25, 30, 5, 18, 22, 14, 11, 9, 20, 13, 7, 24, 16, 19, 21, 6, 17, 23, 4, 28, 12, 10, 15, 8, 26, 19]
profits = [20, 15, 12, 10, 30, 35, 8, 22, 25, 18, 14, 11, 25, 17, 9, 28, 20, 23, 26, 8, 21, 27, 5, 32, 16, 13, 19, 11, 31, 24]
capacity = 120
prob = KnapsackInstance(weights, profits, capacity)

Con esta instancia, el resultado era:

'profit': 158,
'weight': 120,



In [1]:
result = knapsack_simulated_annealing(
    instance=prob, 
    max_iterations=150, 
    initial_temp=500000, 
    cooling_rate=0.9999)

In [1]:
result['profit']